In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [2]:
file_path = "pankaj_shah_2026_cv.pdf"

In [3]:
loader = PyPDFLoader(file_path)
pages = loader.load()

len(pages)

4

In [4]:
pages[0].page_content[0:500]


'Pankaj Shah\nPython / Java AI FullStack Engineer\nBhagwati Road ,\xa0 Inaruwa +977 9804016025 pshah9360@gmail.com\n\uf0e1 @ https://www.linkedin.com/in/pankaj-shah-901a98219/ \uf09b @ https://github.com/shahpankaj123\nI am a dedicated and results-driven Python and Java backend developer with a strong foundation in building scalable and e\x00cient\nsystems. At 22, I have gained substantial industry experience through both internship and professional roles. I began my journey at\nNyxusByte Tech as a Django Python Devel'

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators= ["\n\n","\n","."," "]
)

chunks = text_splitter.split_documents(pages)

print(f"Number of chunks: {len(chunks)}")
print(chunks[0].page_content)

Number of chunks: 22
Pankaj Shah
Python / Java AI FullStack Engineer
Bhagwati Road ,  Inaruwa +977 9804016025 pshah9360@gmail.com
 @ https://www.linkedin.com/in/pankaj-shah-901a98219/  @ https://github.com/shahpankaj123
I am a dedicated and results-driven Python and Java backend developer with a strong foundation in building scalable and e cient
systems. At 22, I have gained substantial industry experience through both internship and professional roles. I began my journey at


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/Users/pankajshah/Desktop/gen_ai/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 13416.98it/s]


In [13]:
from langchain_chroma import Chroma

In [14]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [16]:
print(vectorstore._collection.count())

22


In [17]:
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k": 3})


In [18]:
query = "Give me My github Account"

docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)

maintainable code reects my commitment to excellence. I am passionate about backend development and AI, and I continuously
strive to build robust software solutions that solve real-world problems.
EXPERIENCE
FullStack Engineer INDIA
Avarthan Innovations Mar, 2024 - Present
At Avathan Innovations, I work as a Python/Java Full Stack Engineer, primarily involved in designing and developing scalable web
Pankaj Shah
Python / Java AI FullStack Engineer
Bhagwati Road ,  Inaruwa +977 9804016025 pshah9360@gmail.com
 @ https://www.linkedin.com/in/pankaj-shah-901a98219/  @ https://github.com/shahpankaj123
I am a dedicated and results-driven Python and Java backend developer with a strong foundation in building scalable and e cient
systems. At 22, I have gained substantial industry experience through both internship and professional roles. I began my journey at
NyxusByte Tech
https://github.com/shahpankaj123/My-Certicate/blob/master/PankajShahCerticate.png
INTERN OF THE MONTH 2024
NyxusByte Tech

In [43]:
system_prompt = '''You are Pankaj Shah's resume assistant. Answer in first person using only the resume context. Be concise and professional. If information is missing, say it is not available in the resume.
Context:
{context}
Answer:'''

In [44]:
import getpass
import os

if "groq_open_api_key" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key:  ········


In [45]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [46]:
def fetch_doc(question):
    docs = retriever.invoke(question)
    return "\n\n----\n\n".join(doc.page_content for doc in docs)

In [47]:
query = "my bachelor cgpa grade give"

In [48]:
context = fetch_doc(query)
prompt = system_prompt.format(context=context)
response = llm.invoke(prompt)

print(response.content)


I am a top-performing student at Nepal College of Information Technology (NCIT), consistently ranking first in every semester. I have been recognized on the Dean’s List by Pokhara University for outstanding academic performance and have received scholarships for my Plus2 and Bachelor’s programs due to my top rankings. My technical skills include Python, Java, Data Analysis, HTML/CSS/JavaScript, PHP, MySQL, and frameworks like Django, Flask, FastAPI, and Spring Boot. I also hold a Django Intern Certificate from NyxusByte Tech (2024). Specific work experience or additional certifications beyond those listed are not detailed in my resume.


In [49]:
from langchain_core.prompts import ChatPromptTemplate

messages = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Pankaj Shah's resume assistant.
Answer in first person using only the resume context.
Be concise and professional.
If information is missing, say it is not available in the resume.

Context:
{context}

Answer:
"""
    ),
    ("human", "{question}")
])

In [50]:
from langchain_core.runnables import RunnableLambda

chain = (
    RunnableLambda(lambda q: {
        "question": q,
        "context": fetch_doc(q)
    })
    | messages
    | llm
)

In [51]:
response = chain.invoke("What is your experience in .NET?")
print(response.content)


The resume does not mention any experience with .NET or related technologies. I do not have information on .NET experience in the provided context.


In [52]:
response = chain.invoke("What is your experience in python?")
print(response.content)

I have extensive experience in Python, primarily as a backend developer and data analyst. My work includes building scalable web applications using frameworks like Django and Flask, such as a Django-based e-commerce platform and a Flask-based API for a logistics company. I’ve also utilized Python for data analysis and machine learning projects, leveraging libraries like Pandas, NumPy, and scikit-learn. My resume highlights Python as a core skill for full-stack and AI-driven development.


In [53]:
response = chain.invoke("Where did i work .give me past experience")
print(response.content)

The resume provided does not include specific details about past work experience. It highlights your current role as a Python/Java AI FullStack Engineer but does not list previous employers or positions. If you have additional sections of your resume available, I can help extract that information.


In [54]:
response = chain.invoke("my current job company")
print(response.content)

The information about your current job company is not available in the provided resume context. Let me know if you'd like assistance with other details!


In [55]:
response = chain.invoke("What is my github account?")
print(response.content)

Your GitHub account is **shahpankaj123**, as referenced in the URLs provided in your resume context.


In [56]:
response = chain.invoke("What is my phone Number?")
print(response.content)

Your phone number is not available in the resume context provided.


In [57]:
response = chain.invoke("What is my Address?")
print(response.content)

The resume does not include your address. This information is not available in the provided context.


In [58]:
response = chain.invoke("What is my education along with gpa?")
print(response.content)

Your education and GPA are as follows:  
- **Bachelor of Engineering in Information Technology (BEIT)**, Nepal College of Information Technology (Balkumari, Lalitpur), Jan 2021 – Jul 2025, **GPA: 3.74**.  
- Another GPA of **3.72** is listed in the resume, but the corresponding program/degree is not explicitly mentioned.  

Additionally, you were recognized on the **Dean’s List** at Pokhara University for outstanding academic performance (specific GPA not provided).


In [59]:
response = chain.invoke("What are my project listed in resume?")
print(response.content)

The resume does not explicitly list specific projects. It highlights your role as a FullStack Engineer at Avarthan Innovations (Mar 2024–Present), where you design and develop scalable web applications using Python/Java, but no individual projects are detailed in the provided context.


In [60]:
response = chain.invoke("What are my achievents?")
print(response.content)

Your achievements include:  
1. **Top Performer**: Recognized as the top student in every semester at Nepal College of Information Technology (2025), earning a reward of 12k.  
2. **FullStack Engineer Role**: Currently working as a Python/Java Full Stack Engineer at Avarthan Innovations (Mar 2024–Present), focusing on designing scalable web solutions.  

No additional awards or projects are detailed in the resume.


In [61]:
response = chain.invoke("What my short introductions")
print(response.content)

I am a full-stack developer and educator with expertise in microservices, databases (PostgreSQL, MySQL, MongoDB), machine learning, and LLM applications (LangChain, LangGraph). I have developed projects like a Personalized News Recommendation System and an LLM-powered HR hiring system. I tutor Information Systems Architecture and Full-Stack Development at Biratnagar International College, while also building web applications with Django and React.js. Proficient in English, Maithili, Nepali, and Hindi.
